In [5]:
import numpy as np  # Import NumPy for numerical calculations and arrays.
import statsmodels.api as sm  # Import statsmodels for fitting ordinary least-squares regression models.
from ISLP import load_data  # Import ISLP's function for loading textbook datasets.
from ISLP.models import (ModelSpec as MS ,  # Import ModelSpec and give it the shorter name MS.
summarize ,  # Import summarize to display model coefficient summaries.
poly)  # Import poly for creating polynomial predictors.
from sklearn.model_selection import train_test_split  # Import the function used to split data into training and validation sets.
from functools import partial  # Import partial to create functions with some arguments already fixed.
from sklearn.model_selection import \  # Begin importing cross-validation utilities from scikit-learn.
(cross_validate ,  # Import cross_validate to evaluate a model using cross-validation.
KFold ,  # Import KFold for K-fold cross-validation.
ShuffleSplit)  # Import ShuffleSplit for repeated random train/test splits.
from sklearn.base import clone  # Import clone so model specifications can be copied safely.
from ISLP.models import sklearn_sm  # Import sklearn_sm to make statsmodels models compatible with scikit-learn validation tools.

In [7]:
Auto = load_data('Auto')  # Load the Auto dataset.
Auto_train , Auto_valid = train_test_split (Auto ,  # Randomly divide Auto into training and validation datasets.
test_size =196,  # Use 196 observations as the test set.
random_state =0)  # Make the random split reproducible.

In [11]:
hp_mm = MS(['horsepower'])  # Create a model specification using horsepower as the predictor.
X_train = hp_mm.fit_transform(Auto_train)  # Build the training design matrix from horsepower, including the intercept.
y_train = Auto_train['mpg']  # Select miles per gallon (mpg) as the response variable.
model = sm.OLS(y_train , X_train)  # Create an ordinary least-squares regression model.
results = model.fit()  # Fit the linear regression model to the training data.

In [13]:
X_valid = hp_mm.transform(Auto_valid)  # Transform the validation data using the same model specification.
y_valid = Auto_valid['mpg']  # Get the actual mpg values in the validation set.
valid_pred = results.predict(X_valid)  # Predict mpg for the validation observations.
np.mean (( y_valid - valid_pred)**2)  # Calculate the validation mean squared error (MSE).

23.61661706966988

In [21]:
def evalMSE(terms ,  # Define a reusable function for calculating test/validation MSE.
response ,  # Specify the name of the response variable.
train ,  # Specify the training dataset.
test):  # Specify the test/validation dataset.
    mm = MS(terms)  # Create a model specification from the supplied predictor terms.
    X_train = mm.fit_transform(train)  # Construct the training design matrix.
    y_train = train[response]  # Extract the response from the training data.
    X_test = mm.transform(test)  # Construct the test design matrix using the fitted specification.
    y_test = test[response]  # Extract the response from the test data.
    results = sm.OLS(y_train , X_train).fit()  # Fit OLS regression using the training data.
    test_pred = results.predict(X_test)  # Generate predictions for the test data.
    return np.mean (( y_test - test_pred)**2)  # Return the test-set mean squared error.


In [25]:
MSE = np.zeros(3)  # Create an array to store MSE values for three polynomial degrees.

for idx, degree in enumerate(range(1, 4)):  # Loop through polynomial degrees 1, 2, and 3.
    MSE[idx] = evalMSE(  # Provide the argument/value used by the surrounding operation.
        [poly('horsepower', degree)],  # Use horsepower raised to the current polynomial degree as the predictor.
        'mpg',  # Use mpg as the response.
        Auto_train,  # Use the training portion of Auto.
        Auto_valid  # Use the validation portion of Auto.
    )  # Close the function call or expression.

MSE  # Display the validation MSE for each polynomial degree.

array([23.61661707, 18.76303135, 18.79694163])

In [29]:
Auto_train, Auto_valid = train_test_split(  # Create a new train/validation split.
    Auto,  # Bootstrap from the complete Auto dataset.
    test_size=196,  # Provide the argument/value used by the surrounding operation.
    random_state=3  # Use a different seed to demonstrate that results depend on the split.
)  # Close the function call or expression.

MSE = np.zeros(3)  # Create an array to store MSE values for three polynomial degrees.

for idx, degree in enumerate(range(1, 4)):  # Loop through polynomial degrees 1, 2, and 3.
    MSE[idx] = evalMSE(  # Provide the argument/value used by the surrounding operation.
        [poly('horsepower', degree)],  # Use horsepower raised to the current polynomial degree as the predictor.
        'mpg',  # Use mpg as the response.
        Auto_train,  # Use the training portion of Auto.
        Auto_valid  # Use the validation portion of Auto.
    )  # Close the function call or expression.

MSE  # Display the validation MSE for each polynomial degree.

array([20.75540796, 16.94510676, 16.97437833])

In [35]:
hp_model = sklearn_sm(sm.OLS ,  # Create a scikit-learn-compatible OLS model.
MS(['horsepower']))  # Specify horsepower as the predictor.
X, Y = Auto.drop(columns =['mpg']), Auto['mpg']  # Separate predictors X from the response Y=mpg.
cv_results = cross_validate(hp_model ,  # Perform cross-validation for the horsepower model.
X,  # Supply the predictor matrix.
Y,  # Supply the response vector.
cv=Auto.shape [0])  # Use leave-one-out cross-validation: number of folds equals the number of observations.
cv_err = np.mean(cv_results['test_score'])  # Average the cross-validation test scores.
cv_err  # Display the estimated cross-validation error.

24.23151351792922

In [39]:
cv_error = np.zeros(5)  # Create storage for cross-validation errors for five polynomial degrees.

H = np.array(Auto['horsepower'])  # Convert horsepower to a NumPy array.
M = sklearn_sm(sm.OLS)  # Create a generic scikit-learn-compatible OLS estimator.

for i, d in enumerate(range(1, 6)):  # Evaluate polynomial models of degrees 1 through 5.
    X = np.power.outer(H, np.arange(d + 1))  # Create polynomial columns from horsepower, including powers 0 through d.

    M_CV = cross_validate(  # Run cross-validation for the current polynomial model.
        M,  # Provide the argument/value used by the surrounding operation.
        X,  # Supply the predictor matrix.
        Y,  # Supply the response vector.
        cv=Auto.shape[0]  # Again use leave-one-out cross-validation.
    )  # Close the function call or expression.

    cv_error[i] = np.mean(M_CV['test_score'])  # Store the mean cross-validation error for this degree.

cv_error  # Display cross-validation errors for all five degrees.

array([24.23151352, 19.24821312, 19.33498406, 19.4244303 , 19.03322528])

In [41]:
A = np.array ([3, 5, 9])  # Create a small example array A.
B = np.array ([2, 4])  # Create a small example array B.
np.add.outer(A, B)  # Compute every pairwise sum between A and B.

array([[ 5,  7],
       [ 7,  9],
       [11, 13]])

In [45]:
cv_error = np.zeros(5)  # Create storage for cross-validation errors for five polynomial degrees.

cv = KFold(  # Create a 10-fold cross-validation splitter.
    n_splits=10,  # Divide the data into 10 folds.
    shuffle=True,  # Shuffle observations before creating the folds.
    random_state=0  # Make the shuffled folds reproducible.
)  # use same splits for each degree

for i, d in enumerate(range(1, 6)):  # Evaluate polynomial models of degrees 1 through 5.
    X = np.power.outer(H, np.arange(d + 1))  # Create polynomial columns from horsepower, including powers 0 through d.

    M_CV = cross_validate(  # Run cross-validation for the current polynomial model.
        M,  # Provide the argument/value used by the surrounding operation.
        X,  # Supply the predictor matrix.
        Y,  # Supply the response vector.
        cv=cv  # Use the predefined 10-fold splitter.
    )  # Close the function call or expression.

    cv_error[i] = np.mean(M_CV['test_score'])  # Store the mean cross-validation error for this degree.

cv_error  # Display cross-validation errors for all five degrees.

array([24.20766449, 19.18533142, 19.27626666, 19.47848402, 19.13719048])

In [49]:
validation = ShuffleSplit(n_splits =1,  # Create one random train/test split for validation.
test_size =196,  # Use 196 observations as the test set.
random_state =0)  # Make the random split reproducible.
results = cross_validate(hp_model ,  # Evaluate the horsepower model on the random split.
Auto.drop (['mpg'], axis =1),  # Use all Auto columns except mpg as predictors.
Auto['mpg'],  # Use mpg as the response.
cv=validation);  # Use the ShuffleSplit object to define the validation split.
results['test_score']  # Display the test/validation error from the split.

array([23.61661707])

In [51]:
validation = ShuffleSplit(n_splits =10,  # Create ten repeated random train/test splits.
test_size =196,  # Use 196 observations as the test set.
random_state =0)  # Make the random split reproducible.
results = cross_validate(hp_model ,  # Evaluate the horsepower model on the random split.
Auto.drop (['mpg'], axis =1),  # Use all Auto columns except mpg as predictors.
Auto['mpg'],  # Use mpg as the response.
cv=validation)  # Provide the argument/value used by the surrounding operation.
results['test_score']. mean (), results['test_score'].std()  # Calculate the mean and standard deviation of the ten validation errors.

(23.802232661034164, 1.4218450941091831)

In [55]:
Portfolio = load_data('Portfolio')  # Load the Portfolio dataset used for the bootstrap example.

def alpha_func(D, idx):  # Define a function that estimates the portfolio allocation parameter alpha.
    cov_ = np.cov(  # Calculate the covariance matrix of X and Y for the selected observations.
        D[['X', 'Y']].loc[idx],  # Select X and Y from the bootstrap sample indices.
        rowvar=False  # Treat columns as variables when calculating the covariance matrix.
    )  # Close the function call or expression.

    return (  # Return the estimated alpha value.
        (cov_[1, 1] - cov_[0, 1]) /  # Compute the numerator of the alpha formula.
        (cov_[0, 0] + cov_[1, 1] - 2 * cov_[0, 1])  # Compute the denominator of the alpha formula.
    )  # Close the function call or expression.

In [57]:
alpha_func(Portfolio , range (100))  # Calculate alpha using the first 100 Portfolio observations.

0.57583207459283

In [59]:
rng = np.random.default_rng (0)  # Create a reproducible random-number generator.
alpha_func(Portfolio ,  # Calculate alpha for a bootstrap sample.
rng.choice (100,  # Randomly choose observation indices from 0 through 99.
100,  # Select 100 observations.
replace=True))  # Provide the argument/value used by the surrounding operation.

0.6074452469619004

In [63]:
def boot_SE(func, D, n=None, B=1000, seed=0):  # Define a function that estimates the bootstrap standard error of a statistic.
    rng = np.random.default_rng(seed)  # Create a reproducible random-number generator using the supplied seed.
    first_, second_ = 0, 0  # Initialize running totals needed to calculate the bootstrap variance.

    n = n or D.shape[0]  # Use the full dataset size unless a bootstrap sample size is supplied.

    for _ in range(B):  # Repeat the bootstrap sampling process B times.
        idx = rng.choice(  # Randomly select indices for one bootstrap sample.
            D.index,  # Sample from the dataset's actual index values.
            n,  # Select n observations for the bootstrap sample.
            replace=True  # Provide the argument/value used by the surrounding operation.
        )  # Close the function call or expression.

        value = func(D, idx)  # Calculate the statistic for the current bootstrap sample.

        first_ += value  # Accumulate the statistic values.
        second_ += value ** 2  # Accumulate squared statistic values.

    return np.sqrt(  # Return the estimated bootstrap standard error.
        second_ / B - (first_ / B) ** 2  # Compute the bootstrap variance and take its square root.
    )  # Close the function call or expression.

In [65]:
alpha_SE = boot_SE(alpha_func ,  # Estimate the standard error of alpha using the bootstrap.
Portfolio ,  # Use the Portfolio dataset.
B=1000 ,  # Use 1,000 bootstrap replications.
seed =0)  # Use a fixed seed for reproducibility.
alpha_SE  # Display the bootstrap standard error of alpha.

0.09118176521277699

In [69]:
def boot_OLS(model_matrix, response, D, idx):  # Define a function that fits OLS on a bootstrap sample and returns its coefficients.
    D_ = D.loc[idx]  # Create the bootstrap dataset using the sampled indices.
    Y_ = D_[response]  # Extract the response variable from the bootstrap sample.
    X_ = clone(model_matrix).fit_transform(D_)  # Clone the model specification and build the bootstrap design matrix.

    return sm.OLS(Y_, X_).fit().params  # Fit OLS and return the estimated regression coefficients.

In [71]:
hp_func = partial(boot_OLS , MS(['horsepower']), 'mpg')  # Create a bootstrap function specifically for mpg predicted by horsepower.

In [75]:
rng = np.random.default_rng(0)  # Provide the argument/value used by the surrounding operation.

np.array([  # Collect coefficient estimates from multiple bootstrap samples into an array.
    boot_OLS(  # Fit the OLS model to one bootstrap sample.
        MS(['horsepower']),  # Use horsepower as the predictor.
        'mpg',  # Use mpg as the response.
        Auto,  # Bootstrap from the complete Auto dataset.
        rng.choice(Auto.index, len(Auto), replace=True)  # Generate one bootstrap sample by sampling Auto indices with replacement.
    )  # Close the function call or expression.
    for _ in range(10)  # Repeat the bootstrap regression 10 times to inspect coefficient variation.
])  # Provide the argument/value used by the surrounding operation.

array([[39.12226577, -0.1555926 ],
       [37.18648613, -0.13915813],
       [37.46989244, -0.14112749],
       [38.56723252, -0.14830116],
       [38.95495707, -0.15315141],
       [39.12563927, -0.15261044],
       [38.45763251, -0.14767251],
       [38.43372587, -0.15019447],
       [37.87581142, -0.1409544 ],
       [37.95949036, -0.1451333 ]])

In [77]:
hp_se = boot_SE(hp_func ,  # Estimate bootstrap standard errors for the horsepower regression coefficients.
Auto ,  # Provide the argument/value used by the surrounding operation.
B=1000 ,  # Use 1,000 bootstrap replications.
seed =10)  # Set the random seed for reproducibility.
hp_se  # Display the bootstrap standard errors.

intercept     0.731176
horsepower    0.006092
dtype: float64

In [79]:
hp_model.fit(Auto , Auto['mpg'])  # Fit the horsepower regression to the full Auto dataset.
model_se = summarize(hp_model.results_)['std err']  # Extract the model-based standard errors of the regression coefficients.
model_se  # Display the conventional OLS standard errors.

intercept     0.717
horsepower    0.006
Name: std err, dtype: float64

In [83]:
quad_model = MS([ poly('horsepower', 2, raw=True)])  # Create a quadratic regression specification using horsepower and horsepower squared.
quad_func = partial(boot_OLS ,  # Create a bootstrap function for the quadratic regression.
quad_model ,  # Use the quadratic model specification.
'mpg')  # Use mpg as the response variable.
boot_SE(quad_func , Auto , B=1000)  # Estimate bootstrap standard errors for the quadratic model coefficients using 1,000 samples.

intercept                                  1.538641
poly(horsepower, degree=2, raw=True)[0]    0.024696
poly(horsepower, degree=2, raw=True)[1]    0.000090
dtype: float64